<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/3.3/3.3.1/40_Expanded_Backtest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ======================================================================
# Notebook 40 — Expanded Backtest (Run 3.3.1) — FULL, FIXED
# ======================================================================

from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
import math, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# --------------------------- Paths ------------------------------------
BASE_DIR   = Path("/content/drive/MyDrive/gt-markets")
SRC_RUN    = BASE_DIR / "outputs" / "runs" / "3.3"       # predictions + cached inputs
SRC_SUM    = SRC_RUN / "60_summary"
SRC_INPUTS = SRC_RUN / "00_meta" / "input"

BT_RUN     = BASE_DIR / "outputs" / "runs" / "3.3.1"
BT_SUM     = BT_RUN / "60_summary"
BT_SUM.mkdir(parents=True, exist_ok=True)

print("Reading predictions from :", SRC_SUM)
print("Reading inputs from      :", SRC_INPUTS)
print("Writing outputs to       :", BT_SUM)

# --------------------- Load predictions (clean) ------------------------
preds = pd.read_csv(SRC_SUM / "predictions_master.csv", low_memory=False)
preds["date"] = pd.to_datetime(preds["date"], errors="coerce")
preds = preds.dropna(subset=["date","asset","freq","dataset","model","task"])
preds["asset"] = preds["asset"].astype(str)
preds["freq"] = preds["freq"].astype(str)
preds["dataset"] = preds["dataset"].astype(str)
preds["model"] = preds["model"].astype(str)
preds["task"] = preds["task"].astype(str)

ASSETS   = ["BTC","GOLD","OIL","USDCNY"]
FREQS    = ["D","W"]
DATASETS = ["base","eng","ext"]

print("Predictions loaded:", preds.shape)

# --------------- Load engineered OHLC (for TA + returns) ---------------
eng_files = sorted(SRC_INPUTS.glob("merged_financial_trends_engineered_*.csv"))
assert eng_files, "No engineered input found in 3.3/00_meta/input/"
ENG_FILE = eng_files[-1]
px_full  = pd.read_csv(ENG_FILE, parse_dates=["Date"]).set_index("Date").sort_index()
print("ENG OHLC file used:", ENG_FILE.name, "| shape:", px_full.shape)

# -------------------- Column maps for ENG file -------------------------
ASSET_CLOSE_COLS = {"BTC":"BTC-USD Close","GOLD":"GC=F Close","OIL":"CL=F Close","USDCNY":"USDCNY=X Close"}
ASSET_HIGH_COLS  = {"BTC":"BTC-USD High","GOLD":"GC=F High","OIL":"CL=F High","USDCNY":"USDCNY=X High"}
ASSET_LOW_COLS   = {"BTC":"BTC-USD Low","GOLD":"GC=F Low","OIL":"CL=F Low","USDCNY":"USDCNY=X Low"}

def resample_ohlc(df: pd.DataFrame, asset: str, freq: str) -> pd.DataFrame:
    """Resample ENG OHLC to D/W and compute next-period return (percentage points)."""
    cols = [ASSET_CLOSE_COLS[asset], ASSET_HIGH_COLS[asset], ASSET_LOW_COLS[asset]]
    sub = df[cols].copy()
    res = sub.resample("D" if freq=="D" else "W").last()
    res.columns = ["Close","High","Low"]
    res["y_true_ret_pp"] = res["Close"].pct_change().shift(-1) * 100.0
    return res

# --------------------------- TA builders -------------------------------
def SMA(x, n): return x.rolling(n, min_periods=n).mean()

def RSI(x, n=14):
    d = x.diff()
    up = d.clip(lower=0.0)
    dn = -d.clip(upper=0.0)
    rs = up.rolling(n, min_periods=n).mean() / dn.rolling(n, min_periods=n).mean().replace(0.0, np.nan)
    return 100.0 - 100.0/(1.0 + rs)

def MACD(x, fast=12, slow=26, sig=9):
    ef = x.ewm(span=fast, adjust=False).mean()
    es = x.ewm(span=slow, adjust=False).mean()
    m  = ef - es
    ms = m.ewm(span=sig, adjust=False).mean()
    return m, ms

def ta_positions(df: pd.DataFrame, rsi_cfgs, ma_cfgs, macd_cfgs) -> dict:
    """Generate TA positions ∈{-1,+1} using supplied config grids."""
    c = df["Close"].astype(float)
    pos = {}

    # RSI contrarian
    for p, ob, os in rsi_cfgs:
        r = RSI(c, p)
        name = f"TA_RSI{p}_{ob}-{os}"
        s = pd.Series(0.0, index=df.index)
        s[r <= os] = 1.0
        s[r >= ob] = -1.0
        pos[name] = s

    # MA cross
    for f, s in ma_cfgs:
        name = f"TA_MAcross_{f}-{s}"
        sf = SMA(c, f); ss = SMA(c, s)
        sgn = pd.Series(0.0, index=df.index)
        sgn[sf > ss] = 1.0
        sgn[sf < ss] = -1.0
        pos[name] = sgn

    # MACD
    for f, s, sg in macd_cfgs:
        name = f"TA_MACD_{f}-{s}-{sg}"
        m, ms = MACD(c, f, s, sg)
        sgn = pd.Series(0.0, index=df.index)
        sgn[m > ms] = 1.0
        sgn[m < ms] = -1.0
        pos[name] = sgn

    return pos

# ----------------------- Strategy primitives --------------------------
COST_BPS = {"BTC":10,"GOLD":3,"OIL":3,"USDCNY":1}
AF = {"D":252, "W":52}

CLS_THRESHOLDS = [(0.52,0.48,"prob052"), (0.55,0.45,"prob055"), (0.60,0.40,"prob060")]
BIGMOVE = {"D":[0.005,0.010,0.015], "W":[0.015,0.020,0.030]}  # % thresholds
REG_K = [2.0, 3.0]
HYB_ALPHA = [0.25, 0.50, 0.75]

RSI_CONFIGS  = [(14,70,30),(14,65,35),(21,70,30),(21,65,35)]
MA_CONFIGS   = [(10,50),(20,100),(50,200)]
MACD_CONFIGS = [(12,26,9),(19,39,9)]

def trading_cost_series(pos: pd.Series, asset: str) -> pd.Series:
    dpos = pos.diff().abs().fillna(pos.abs())
    return (COST_BPS[asset] / 10000.0) * dpos

def simulate_pnl(dates, pos: pd.Series, y_true_ret_pp: pd.Series, asset: str, freq: str) -> pd.Series:
    pos  = pos.reindex(dates).fillna(0.0)
    rets = (y_true_ret_pp.reindex(dates).fillna(0.0)) / 100.0
    costs = trading_cost_series(pos, asset)
    return pos * rets - costs

def ann_return(pnl: pd.Series, freq: str) -> float:
    if len(pnl) == 0: return 0.0
    growth = (1.0 + pnl).prod()
    return float(growth ** (AF[freq] / max(len(pnl), 1)) - 1.0)

def sharpe(pnl: pd.Series, freq: str) -> float:
    vol = pnl.std(ddof=0)
    if vol == 0.0 or len(pnl) < 2: return 0.0
    return float(pnl.mean() / vol * math.sqrt(AF[freq]))

def max_dd(pnl: pd.Series) -> float:
    eq = (1.0 + pnl).cumprod()
    dd = (eq / eq.cummax()) - 1.0
    return float(dd.min())

# ---------------------- Pred aggregation helpers ----------------------
def agg_cls_prob(sub: pd.DataFrame, model: str) -> pd.Series:
    return (sub[(sub["task"]=="classification") & (sub["model"]==model)]
              .dropna(subset=["prob_up"])
              .groupby("date")["prob_up"].mean().sort_index())

def agg_reg_pred(sub: pd.DataFrame, model: str) -> pd.Series:
    return (sub[(sub["task"]=="regression") & (sub["model"]==model)]
              .dropna(subset=["pred_ret"])
              .groupby("date")["pred_ret"].mean().sort_index())

def ensemble_pred_ret(sub: pd.DataFrame) -> pd.Series:
    return (sub[sub["task"]=="regression"]
              .dropna(subset=["pred_ret"])
              .groupby("date")["pred_ret"].median().sort_index())

def cls_position(prob: pd.Series, hi: float, lo: float) -> pd.Series:
    pos = pd.Series(0.0, index=prob.index)
    pos[prob >= hi] = 1.0
    pos[prob <= lo] = -1.0
    return pos

def reg_sized_position(pred_pp: pd.Series, k: float) -> pd.Series:
    return (pred_pp / k).clip(-1.0, 1.0)

def apply_bigmove_filter(pos: pd.Series, pred_pp: pd.Series, tau: float) -> pd.Series:
    # tau is in %, pred_pp in percentage points → compare to tau*100
    return pos.where(pred_pp.abs() >= tau*100.0, 0.0)

def hybrid_confirm(pos_ml: pd.Series, pos_ta: pd.Series) -> pd.Series:
    return pos_ml.where(np.sign(pos_ml) == np.sign(pos_ta), 0.0)

def hybrid_weight(pos_ml: pd.Series, pos_ta: pd.Series, alpha: float) -> pd.Series:
    return (alpha*pos_ml + (1.0-alpha)*pos_ta).clip(-1.0, 1.0)

# ------------------------ Backtest Enumeration ------------------------
records = []
config_rows = []

for dataset in DATASETS:
    for asset in ASSETS:
        for freq in FREQS:
            sub = preds[(preds["dataset"]==dataset) & (preds["asset"]==asset) & (preds["freq"]==freq)]
            if sub.empty:
                continue

            # Build trading calendar
            px = resample_ohlc(px_full, asset, freq)
            dates_pred = pd.DatetimeIndex(sorted(sub["date"].unique()))
            dates = px.index.intersection(dates_pred)
            if len(dates) < 50:
                continue
            px = px.reindex(dates).ffill()
            y_true_ret_pp = px["y_true_ret_pp"]

            # TA positions for this asset/freq
            ta_pos = ta_positions(px, RSI_CONFIGS, MA_CONFIGS, MACD_CONFIGS)
            for k in ta_pos:
                ta_pos[k] = ta_pos[k].reindex(dates).fillna(0.0)

            # Ensemble magnitude for CLS big-move filters
            pred_ret_ens = ensemble_pred_ret(sub).reindex(dates)

            # A) ML — Classification (threshold & big-move)
            models_cls = sub.loc[sub["task"]=="classification","model"].dropna().unique().tolist()
            for m in models_cls:
                prob = agg_cls_prob(sub, m).reindex(dates).fillna(method="ffill").fillna(0.5).clip(0,1)
                for hi, lo, tag in CLS_THRESHOLDS:
                    pos_base = cls_position(prob, hi, lo)
                    # no big-move
                    pnl = simulate_pnl(dates, pos_base, y_true_ret_pp, asset, freq)
                    sid = f"ML_CLS_{m}_{tag}"
                    records.append([dataset,asset,freq,sid,"ML_CLS",ann_return(pnl,freq),sharpe(pnl,freq),max_dd(pnl)])
                    config_rows.append({"strategy_id":sid,"family":"ML_CLS",
                                        "params":json.dumps({"model":m,"hi":hi,"lo":lo,"bm":None})})
                    # with big-move
                    for tau in BIGMOVE[freq]:
                        if pred_ret_ens.isna().all():
                            continue
                        pos_bm = apply_bigmove_filter(pos_base, pred_ret_ens, tau)
                        pnl_bm = simulate_pnl(dates, pos_bm, y_true_ret_pp, asset, freq)
                        sid_bm = f"ML_CLS_{m}_{tag}_bm{int(tau*1000)/10:.1f}p"
                        records.append([dataset,asset,freq,sid_bm,"ML_CLS",ann_return(pnl_bm,freq),sharpe(pnl_bm,freq),max_dd(pnl_bm)])
                        config_rows.append({"strategy_id":sid_bm,"family":"ML_CLS",
                                            "params":json.dumps({"model":m,"hi":hi,"lo":lo,"bm":tau})})

            # B) ML — Regression sized
            models_reg = sub.loc[sub["task"]=="regression","model"].dropna().unique().tolist()
            for m in models_reg:
                pred_pp = agg_reg_pred(sub, m).reindex(dates).fillna(0.0)
                for k_val in REG_K:
                    pos = reg_sized_position(pred_pp, k_val)
                    pnl = simulate_pnl(dates, pos, y_true_ret_pp, asset, freq)
                    sid = f"ML_REG_{m}_k{str(k_val).replace('.','p')}"
                    records.append([dataset,asset,freq,sid,"ML_REG",ann_return(pnl,freq),sharpe(pnl,freq),max_dd(pnl)])
                    config_rows.append({"strategy_id":sid,"family":"ML_REG",
                                        "params":json.dumps({"model":m,"k":k_val})})

            # C) TA-only (generate names from configs; do not read from preds)
            for ta_name, pos_ta in ta_pos.items():
                pnl = simulate_pnl(dates, pos_ta, y_true_ret_pp, asset, freq)
                records.append([dataset,asset,freq,ta_name,"TA_ONLY",ann_return(pnl,freq),sharpe(pnl,freq),max_dd(pnl)])

                # Safe parsing for config_reference
                params = {}
                if ta_name.startswith("TA_RSI"):
                    parts = ta_name.split("_")               # ["TA","RSI14","70-30"]
                    win = int(parts[1].replace("RSI",""))
                    levels = parts[2]
                    params = {"type":"RSI","win":win,"levels":levels}
                elif ta_name.startswith("TA_MAcross"):
                    parts = ta_name.split("_")               # ["TA","MAcross","10-50"]
                    fast, slow = parts[2].split("-")
                    params = {"type":"MA","fast":int(fast),"slow":int(slow)}
                elif ta_name.startswith("TA_MACD"):
                    parts = ta_name.split("_")               # ["TA","MACD","12-26-9"]
                    f, s, sg = parts[2].split("-")
                    params = {"type":"MACD","fast":int(f),"slow":int(s),"signal":int(sg)}
                config_rows.append({"strategy_id":ta_name,"family":"TA_ONLY","params":json.dumps(params)})

            # D) Hybrids — Confirm (CLS + TA agree)
            for m in models_cls:
                prob = agg_cls_prob(sub, m).reindex(dates).fillna(method="ffill").fillna(0.5).clip(0,1)
                for hi, lo, tag in CLS_THRESHOLDS:
                    pos_ml = cls_position(prob, hi, lo)
                    for ta_name, pos_ta in ta_pos.items():
                        pos = hybrid_confirm(pos_ml, pos_ta)
                        pnl = simulate_pnl(dates, pos, y_true_ret_pp, asset, freq)
                        sid = f"HYB_CONF_{m}_{ta_name}_{tag}"
                        records.append([dataset,asset,freq,sid,"HYBRID_CONF",ann_return(pnl,freq),sharpe(pnl,freq),max_dd(pnl)])
                        config_rows.append({"strategy_id":sid,"family":"HYBRID_CONF",
                                            "params":json.dumps({"model":m,"hi":hi,"lo":lo,"ta":ta_name})})

            # E) Hybrids — Weighted (REG-sized ML + TA)
            for m in models_reg:
                pred_pp = agg_reg_pred(sub, m).reindex(dates).fillna(0.0)
                base_pos_ml = reg_sized_position(pred_pp, REG_K[0])  # baseline k
                for ta_name, pos_ta in ta_pos.items():
                    for a in HYB_ALPHA:
                        pos = hybrid_weight(base_pos_ml, pos_ta, a)
                        pnl = simulate_pnl(dates, pos, y_true_ret_pp, asset, freq)
                        sid = f"HYB_WGT_{m}_{ta_name}_a{str(a).replace('.','p')}"
                        records.append([dataset,asset,freq,sid,"HYBRID_WEIGHT",ann_return(pnl,freq),sharpe(pnl,freq),max_dd(pnl)])
                        config_rows.append({"strategy_id":sid,"family":"HYBRID_WEIGHT",
                                            "params":json.dumps({"model":m,"alpha":a,"ta":ta_name,"k":REG_K[0]})})

# -------------------------- Save outputs -------------------------------
cols = ["dataset","asset","freq","strategy_id","family","ann_return","sharpe","max_dd"]
performance_master = pd.DataFrame(records, columns=cols)
performance_master = performance_master.sort_values(["dataset","asset","freq","family","strategy_id"]).reset_index(drop=True)
performance_master.to_csv(BT_SUM / "performance_master.csv", index=False)
print("Saved:", BT_SUM / "performance_master.csv", "| rows:", len(performance_master))

leaderboard = (performance_master
               .sort_values(["dataset","asset","freq","sharpe"], ascending=[True,True,True,False])
               .groupby(["dataset","asset","freq"])
               .head(5)
               .reset_index(drop=True))
leaderboard.to_csv(BT_SUM / "strategy_leaderboard.csv", index=False)
print("Saved:", BT_SUM / "strategy_leaderboard.csv", "| rows:", len(leaderboard))

config_ref = pd.DataFrame(config_rows)
config_ref.to_csv(BT_SUM / "config_reference.csv", index=False)
print("Saved:", BT_SUM / "config_reference.csv", "| rows:", len(config_ref))

print("✅ Expanded backtest complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading predictions from : /content/drive/MyDrive/gt-markets/outputs/runs/3.3/60_summary
Reading inputs from      : /content/drive/MyDrive/gt-markets/outputs/runs/3.3/00_meta/input
Writing outputs to       : /content/drive/MyDrive/gt-markets/outputs/runs/3.3.1/60_summary
Predictions loaded: (60000, 12)
ENG OHLC file used: merged_financial_trends_engineered_2025-09-07.csv | shape: (2609, 254)
Saved: /content/drive/MyDrive/gt-markets/outputs/runs/3.3.1/60_summary/performance_master.csv | rows: 3480
Saved: /content/drive/MyDrive/gt-markets/outputs/runs/3.3.1/60_summary/strategy_leaderboard.csv | rows: 120
Saved: /content/drive/MyDrive/gt-markets/outputs/runs/3.3.1/60_summary/config_reference.csv | rows: 3480
✅ Expanded backtest complete.


In [ ]:

# === Generate README for 3.3.1/60_summary ===
from pathlib import Path

# Define output folder explicitly
OUT_DIR = Path("/content/drive/MyDrive/gt-markets/outputs/runs/3.3.1/60_summary")
OUT_DIR.mkdir(parents=True, exist_ok=True)

readme_text = """\
# Backtest Results (3.3.1)

This folder contains the expanded backtest outputs generated from predictions (3.3 summary).
It includes model-based, TA-based, and hybrid strategies across multiple configurations.

---

## Files

- **performance_master.csv**
  Contains full performance metrics for every strategy configuration.
  Columns include:
  - asset, freq, dataset
  - strategy_id (unique config label)
  - annualised_return, sharpe, maxdd, winrate, trades
  - task (classification/regression/ta/hybrid)
  - config_id (link to config_reference)

- **strategy_leaderboard.csv**
  Top strategies per asset × freq × dataset (by Sharpe).
  Useful for quick comparison of best performers.

- **config_reference.csv**
  Lookup table that explains each `strategy_id`.
  Includes:
  - strategy_id
  - family (ML, DL, TA, Hybrid)
  - parameters (thresholds, TA windows, etc.)
  - description

---

## Notes

- Model-based configs tested at thresholds:
  - Classification: 0.52/0.48, 0.55/0.45, 0.60/0.40
  - Regression big-move filters: Daily {0.5%, 1.0%, 1.5%}, Weekly {1.5%, 2.0%, 3.0%}

- TA configs tested:
  - RSI: periods 14 & 21, levels 70/30 and 65/35
  - MA Cross: (10,50), (20,100), (50,200)
  - MACD: (12,26,9), (19,39,9)

- Hybrids:
  - Baseline ML trade + TA confirm filter
  - Weighted positions by confidence

- MaxDD values are reported as decimals (e.g. 0.10 = 10%).

---

Generated: 3.3.1
"""

with open(OUT_DIR / "README.txt", "w") as f:
    f.write(readme_text)

print(f"✅ README written to: {OUT_DIR/'README.txt'}")

✅ README written to: /content/drive/MyDrive/gt-markets/outputs/runs/3.3.1/60_summary/README.txt
